# 01 - Data Ingestion

## Objective

This notebook ingests the monthly BTS flight datasets and the supporting reference datasets from the Databricks Volume.

The ingestion process includes:

- Loading the project configuration
- Reading and consolidating the monthly flight CSV files
- Validating the flight dataset
- Saving the consolidated flight data as a Delta table
- Reading and validating the reference CSV files
- Saving each reference dataset as a reusable Delta lookup table

The raw flight data and reference datasets are stored separately. Dataset enrichment and permanent joins will be performed during the data-cleaning stage.

#### Load project configuration

In [0]:
# Load the project configuration

from config import project_config as cfg
from pyspark.sql import functions as F

print("Project configuration loaded successfully.")

Project configuration loaded successfully.


#### Flight data ingestion

This section reads and consolidates the monthly flight CSV files stored in the raw data directory.

In [0]:
# Load the monthly flight CSV files

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(cfg.RAW_PATH)
)

print(f"Flight data loaded successfully from: {cfg.RAW_PATH}")

Flight data loaded successfully from: /Volumes/workspace/default/flight_delay_capstone/raw


#### Validate flight dataset

In [0]:
# Validate that the consolidated flight dataset is not empty

flight_record_count = df_raw.count()
flight_column_count = len(df_raw.columns)

if flight_record_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset is empty."
    )

if flight_column_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset contains no columns."
    )

print(f"Total flight records: {flight_record_count:,}")
print(f"Total flight columns: {flight_column_count}")

Total flight records: 7,001,619
Total flight columns: 32


#### Preview flight dataset

In [0]:
display(df_raw.limit(10))

QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
3,7,1,7/7/2025 12:00:00 AM,AA,1,JFK,"New York, NY",New York,LAX,"Los Angeles, CA",California,720,-5.0,0.0,18.0,7.0,1022,-26.0,0.0,0.0,null,0.0,362.0,341.0,316.0,2475.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,10,LAX,"Los Angeles, CA",California,JFK,"New York, NY",New York,2121,4.0,0.0,22.0,11.0,559,-14.0,0.0,0.0,null,0.0,338.0,320.0,287.0,2475.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",Wisconsin,CLT,"Charlotte, NC",North Carolina,716,-7.0,0.0,13.0,19.0,1030,-12.0,0.0,0.0,null,0.0,134.0,129.0,97.0,708.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1003,CLT,"Charlotte, NC",North Carolina,MCI,"Kansas City, MO",Missouri,1615,1.0,0.0,71.0,6.0,1737,48.0,1.0,0.0,null,0.0,142.0,189.0,112.0,808.0,0.0,0.0,47.0,0.0,1.0
3,7,1,7/7/2025 12:00:00 AM,AA,1003,MCI,"Kansas City, MO",Missouri,CLT,"Charlotte, NC",North Carolina,1827,38.0,1.0,16.0,19.0,2155,29.0,1.0,0.0,null,0.0,148.0,139.0,104.0,808.0,0.0,0.0,0.0,0.0,29.0
3,7,1,7/7/2025 12:00:00 AM,AA,1004,BOS,"Boston, MA",Massachusetts,DCA,"Washington, DC",Virginia,2106,67.0,1.0,26.0,6.0,2250,63.0,1.0,0.0,null,0.0,104.0,100.0,68.0,399.0,0.0,0.0,0.0,0.0,63.0
3,7,1,7/7/2025 12:00:00 AM,AA,1007,CLT,"Charlotte, NC",North Carolina,PNS,"Pensacola, FL",Florida,2301,-5.0,0.0,33.0,6.0,2354,-10.0,0.0,0.0,null,0.0,113.0,108.0,69.0,488.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1008,ATL,"Atlanta, GA",Georgia,DFW,"Dallas/Fort Worth, TX",Texas,1633,148.0,1.0,23.0,44.0,1801,180.0,1.0,0.0,null,0.0,148.0,180.0,113.0,731.0,0.0,0.0,32.0,0.0,148.0
3,7,1,7/7/2025 12:00:00 AM,AA,1009,LAX,"Los Angeles, CA",California,ORD,"Chicago, IL",Illinois,2259,-5.0,0.0,20.0,4.0,518,-33.0,0.0,0.0,null,0.0,259.0,231.0,207.0,1744.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1010,DFW,"Dallas/Fort Worth, TX",Texas,STL,"St. Louis, MO",Missouri,2105,6.0,0.0,15.0,6.0,2257,-9.0,0.0,0.0,null,0.0,112.0,97.0,76.0,550.0,null,null,null,null,null


#### Review flight schema

In [0]:
# Review the schema inferred from the monthly CSV files

df_raw.printSchema()

root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELA

#### Save raw flight table

In [0]:
# Store the consolidated flight dataset as a Delta table

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.RAW_TABLE)
)

print(f"Raw flight table created successfully: {cfg.RAW_TABLE}")

Raw flight table created successfully: workspace.default.flights_raw


#### Validate saved flight table

In [0]:
# Confirm that the saved Delta table contains the expected records

saved_flight_count = spark.read.table(
    cfg.RAW_TABLE
).count()

if saved_flight_count != flight_record_count:
    raise RuntimeError(
        "The saved flight-table record count does not match "
        "the ingested dataset."
    )

print(
    f"Saved flight-table records confirmed: "
    f"{saved_flight_count:,}"
)

Saved flight-table records confirmed: 7,001,619


## Reference data ingestion

This section reads the supporting reference CSV files and stores each dataset as a separate Delta lookup table.

The reference datasets provide descriptive values for coded flight attributes, including airlines, airports, cancellation reasons, months, quarters, weekdays, and binary indicators.

#### Load and inspect reference datasets

In [0]:
# Load each configured reference CSV file

reference_dataframes = {}
reference_summaries = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_path = reference_config["path"]

    reference_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(reference_path)
    )

    record_count = reference_df.count()
    column_count = len(reference_df.columns)

    if record_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' is empty."
        )

    if column_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' "
            f"contains no columns."
        )

    reference_dataframes[reference_name] = reference_df

    reference_summaries.append(
        (
            reference_name,
            reference_path,
            record_count,
            column_count,
            ", ".join(reference_df.columns),
        )
    )

print(
    f"Reference datasets loaded successfully: "
    f"{len(reference_dataframes)}"
)

Reference datasets loaded successfully: 7


#### Review reference dataset summary

In [0]:
# Create a summary of the loaded reference datasets

reference_summary_df = spark.createDataFrame(
    reference_summaries,
    [
        "reference_name",
        "source_path",
        "record_count",
        "column_count",
        "columns",
    ],
)

display(
    reference_summary_df.orderBy("reference_name")
)

reference_name,source_path,record_count,column_count,columns
airlines,/Volumes/workspace/default/flight_delay_capstone/reference/L_UNIQUE_CARRIERS_Reporting_Airline.csv,1776,2,"Code, Description"
airports,/Volumes/workspace/default/flight_delay_capstone/reference/L_AIRPORT_Origin_Dest.csv,6914,2,"Code, Description"
cancellation_codes,/Volumes/workspace/default/flight_delay_capstone/reference/L_CANCELLATION_CancellationCode.csv,4,2,"Code, Description"
months,/Volumes/workspace/default/flight_delay_capstone/reference/L_MONTHS_Month.csv,12,2,"Code, Description"
quarters,/Volumes/workspace/default/flight_delay_capstone/reference/L_QUARTERS_Quarter.csv,4,2,"Code, Description"
weekdays,/Volumes/workspace/default/flight_delay_capstone/reference/L_WEEKDAYS_DayOfWeek.csv,8,2,"Code, Description"
yes_no,/Volumes/workspace/default/flight_delay_capstone/reference/L_YESNO_RESP_ArrDel15_DepDel15_Cancelled_Diverted.csv,2,2,"Code, Description"


#### Validate reference schemas

In [0]:
# Confirm that each reference dataset contains Code and Description

required_reference_columns = {
    "Code",
    "Description",
}

for reference_name, reference_df in (
    reference_dataframes.items()
):
    available_columns = set(reference_df.columns)

    missing_columns = (
        required_reference_columns
        - available_columns
    )

    if missing_columns:
        raise ValueError(
            f"Reference dataset '{reference_name}' is missing "
            f"the following required columns: "
            f"{sorted(missing_columns)}"
        )

print(
    "All reference datasets contain the required "
    "'Code' and 'Description' columns."
)

All reference datasets contain the required 'Code' and 'Description' columns.


#### Validate reference codes

In [0]:
# Check for null and duplicate codes in each reference dataset

reference_quality_results = []

for reference_name, reference_df in (
    reference_dataframes.items()
):
    total_records = reference_df.count()

    null_code_count = (
        reference_df
        .filter(F.col("Code").isNull())
        .count()
    )

    distinct_code_count = (
        reference_df
        .select("Code")
        .distinct()
        .count()
    )

    duplicate_code_count = (
        total_records
        - distinct_code_count
    )

    reference_quality_results.append(
        (
            reference_name,
            total_records,
            null_code_count,
            duplicate_code_count,
        )
    )

reference_quality_df = spark.createDataFrame(
    reference_quality_results,
    [
        "reference_name",
        "record_count",
        "null_code_count",
        "duplicate_code_count",
    ],
)

display(
    reference_quality_df.orderBy("reference_name")
)

reference_name,record_count,null_code_count,duplicate_code_count
airlines,1776,0,0
airports,6914,0,0
cancellation_codes,4,0,0
months,12,0,0
quarters,4,0,0
weekdays,8,0,0
yes_no,2,0,0


In [0]:
# Stop ingestion if reference keys contain quality issues

invalid_reference_datasets = (
    reference_quality_df
    .filter(
        (F.col("null_code_count") > 0)
        | (F.col("duplicate_code_count") > 0)
    )
    .count()
)

if invalid_reference_datasets > 0:
    raise ValueError(
        "One or more reference datasets contain null or "
        "duplicate codes. Review the quality results before "
        "saving the lookup tables."
    )

print("Reference-code quality validation completed successfully.")

Reference-code quality validation completed successfully.


#### Preview reference datasets

In [0]:
# Display a small sample from each reference dataset

for reference_name, reference_df in (
    reference_dataframes.items()
):
    print(f"Reference dataset: {reference_name}")
    display(reference_df.limit(10))

Reference dataset: airports


Code,Description
01A,"Afognak Lake, AK: Afognak Lake Airport"
03A,"Granite Mountain, AK: Bear Creek Mining Strip"
04A,"Lik, AK: Lik Mining Camp"
05A,"Little Squaw, AK: Little Squaw Airport"
05K,"Port Alsworth, AK: Wilder Runway"
06A,"Kizhuyak, AK: Kizhuyak Bay"
07A,"Klawock, AK: Klawock Seaplane Base"
08A,"Elizabeth Island, AK: Elizabeth Island Airport"
09A,"Homer, AK: Augustin Island"
1AK,"Mertarvik, AK: Mertarvik Quarry Road Landing Strip"


Reference dataset: cancellation_codes


Code,Description
A,Carrier
B,Weather
C,National Air System
D,Security


Reference dataset: months


...

Code,Description
02Q,Titan Airways
05Q,"Comlux Aviation, AG"
06Q,Master Top Linhas Aereas Ltd.
07Q,Flair Airlines Ltd.
09Q,"Swift Air, LLC d/b/a Eastern Air Lines d/b/a Eastern"
0BQ,DCA
0CQ,ACM AIR CHARTER GmbH
0FQ,"Maine Aviation Aircraft Charter, LLC"
0GQ,"Inter Island Airways, d/b/a Inter Island Air"
0HQ,Polar Airlines de Mexico d/b/a Nova Air


Reference dataset: weekdays


Code,Description
1,Monday
2,Tuesday
3,Wednesday
4,Thursday
5,Friday
6,Saturday
7,Sunday
9,Unknown


Reference dataset: yes_no


Code,Description
0,No
1,Yes


#### Save reference lookup tables

In [0]:
# Store each reference dataset as a Delta lookup table

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]
    reference_df = reference_dataframes[reference_name]

    (
        reference_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(reference_table)
    )

    print(
        f"Lookup table created successfully: "
        f"{reference_table}"
    )

Lookup table created successfully: workspace.default.airports_lookup
Lookup table created successfully: workspace.default.cancellation_codes_lookup
Lookup table created successfully: workspace.default.months_lookup
Lookup table created successfully: workspace.default.quarters_lookup
Lookup table created successfully: workspace.default.airlines_lookup
Lookup table created successfully: workspace.default.weekdays_lookup
Lookup table created successfully: workspace.default.yes_no_lookup


#### Validate saved lookup tables

In [0]:
# Confirm that each saved lookup table preserves its record count

lookup_validation_results = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]

    source_count = reference_dataframes[
        reference_name
    ].count()

    saved_count = spark.read.table(
        reference_table
    ).count()

    counts_match = source_count == saved_count

    lookup_validation_results.append(
        (
            reference_name,
            reference_table,
            source_count,
            saved_count,
            counts_match,
        )
    )

lookup_validation_df = spark.createDataFrame(
    lookup_validation_results,
    [
        "reference_name",
        "table_name",
        "source_record_count",
        "saved_record_count",
        "counts_match",
    ],
)

display(
    lookup_validation_df.orderBy("reference_name")
)

reference_name,table_name,source_record_count,saved_record_count,counts_match
airlines,workspace.default.airlines_lookup,1776,1776,true
airports,workspace.default.airports_lookup,6914,6914,true
cancellation_codes,workspace.default.cancellation_codes_lookup,4,4,true
months,workspace.default.months_lookup,12,12,true
quarters,workspace.default.quarters_lookup,4,4,true
weekdays,workspace.default.weekdays_lookup,8,8,true
yes_no,workspace.default.yes_no_lookup,2,2,true


In [0]:
# Stop execution if any lookup-table count does not match

failed_lookup_validations = (
    lookup_validation_df
    .filter(F.col("counts_match") == False)
    .count()
)

if failed_lookup_validations > 0:
    raise RuntimeError(
        "One or more lookup-table record counts do not "
        "match their source datasets."
    )

print("All lookup-table record counts were validated successfully.")

All lookup-table record counts were validated successfully.


#### Data ingestion completion

In [0]:
print("Data ingestion completed successfully.")
print(f"Raw flight table: {cfg.RAW_TABLE}")
print(
    f"Reference lookup tables created: "
    f"{len(cfg.REFERENCE_DATASETS)}"
)

Data ingestion completed successfully.
Raw flight table: workspace.default.flights_raw
Reference lookup tables created: 7
